# Week 7 Development Notebook

## 1. Environment and shared infrastructure

### Design decisions

**State representation.** Integer states (0..499) end-to-end. Q-table is a `numpy.ndarray` of shape (500, 6); model entries are keyed by `(int_state, int_action)`. A `decode_state()` helper returns the (taxi_row, taxi_col, passenger_loc, destination) tuple and is reserved for diagnostics, logging, and post-hoc analysis only, never agent hot paths.

**Environment.** Gymnasium `Taxi-v4` in deterministic mode (default). The project-level environment class wraps `gym.make("Taxi-v4")`, exposes `reset()` and `step(action)` returning integer states alongside `terminated` and `truncated` flags, supports a deterministic seed, and is agnostic to any Gymnasium wrappers stacked beneath it (e.g., the Epic 5 mutation wrapper).

**Episode truncation.** Unsolved episodes truncate at step 200 via Gymnasium's `TimeLimit` wrapper, signaled by `truncated=True` (distinct from `terminated=True` for goal completion). Agents must distinguish the two when computing Q-update targets.

**Agent interface.** Agents expose `act(state)` and `learn(state, action, reward, next_state, terminated, truncated)`. The training-loop driver consumes this interface and is agent-agnostic.

**Trace schema (per real step).** Columns: `agent_id`, `seed`, `step`, `episode`, `step_in_episode`, `state`, `action`, `reward`, `next_state`, `terminated`, `truncated`, `wall_time_step`, `wall_time_planning`, `n_planning_updates`. Hyperparameters live in a separate config sidecar joined by `(agent_id, seed)`.

**Persistence.** One trace CSV per run, written to `data/`, with a matching JSON config sidecar. Trace filenames encode `agent_id` and `seed`. The driver owns I/O; agents never write files. A `load_traces()` helper concatenates matching CSVs into a tidy DataFrame for analysis.

**Storage format.** pandas DataFrame in memory; CSV on disk. In-memory is sufficient for Taxi-scale runs.

**Timing.** The driver wraps each real step to measure `wall_time_step`; the agent reports `wall_time_planning` and `n_planning_updates` back to the driver as part of `learn()`'s return value (or via a side-channel attribute). To be finalized when implementing item 4.

In [1]:
# DONE: Build an environment class that wraps Gymnasium Taxi-v4 and presents
# a clean interface to agents.
# Outcome: a class exposing reset() and step(action) returning integer states,
# rewards, terminated, and truncated; supports a deterministic seed; isolates
# agents from Gymnasium API details; ready to accept Gymnasium wrappers (e.g.,
# the Epic 5 mutation wrapper) without changes to the class itself.

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from taxi_env import TaxiEnv

env = TaxiEnv(seed=42)
state = env.reset()
print(f"Initial state (int): {state}")
print(f"Decoded (taxi_row, taxi_col, passenger_loc, destination): {env.decode_state(state)}")

for step_i in range(5):
    next_state, reward, terminated, truncated = env.step(action=0)
    print(
        f"step={step_i} action=0 -> next_state={next_state} "
        f"reward={reward} terminated={terminated} truncated={truncated} "
        f"decoded={env.decode_state(next_state)}"
    )
    if terminated or truncated:
        break

Initial state (int): 386
Decoded (taxi_row, taxi_col, passenger_loc, destination): (3, 4, 1, 2)
step=0 action=0 -> next_state=486 reward=-1.0 terminated=False truncated=False decoded=(4, 4, 1, 2)
step=1 action=0 -> next_state=486 reward=-1.0 terminated=False truncated=False decoded=(4, 4, 1, 2)
step=2 action=0 -> next_state=486 reward=-1.0 terminated=False truncated=False decoded=(4, 4, 1, 2)
step=3 action=0 -> next_state=486 reward=-1.0 terminated=False truncated=False decoded=(4, 4, 1, 2)
step=4 action=0 -> next_state=486 reward=-1.0 terminated=False truncated=False decoded=(4, 4, 1, 2)


In [ ]:
# TODO: Build a training-loop driver that runs any agent satisfying a known
# interface against the environment class.
# Outcome: a driver that takes an agent exposing act(state) and
# learn(s, a, r, s_prime, terminated, truncated), runs for a configurable
# number of real steps or episodes, and records per-step traces (reward,
# episode index, step count) for downstream analysis. Smoke-tested with a
# random-action agent.

In [ ]:
# TODO: Verify Taxi-v4 episodes truncate at 200 steps for unsolved episodes.
# Outcome: a test that drives the env with actions guaranteed not to solve
# (e.g., never pick up the passenger), confirms the episode ends at step 200,
# and that termination is signaled via truncated=True (not terminated=True).

## 2. Q-learning baseline

## 3. Tabular model and Dyna-Q

## 4. Sample-efficiency visualizations

## 5. Dynamic environment

## 6. Dyna-Q+

## 7. Prioritized sweeping

## 8. Synthesis

## 9. Optional: NN dynamics model